In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when

# 1. START SPARK WITH INCREASED MEMORY (8GB RAM) to prevent OOM errors
# Spark assigns by default only 1GB to driver and executors, which is insufficient for our dataset size and operations.
spark = SparkSession.builder \
    .appName("NYC_Taxi_Stage5_Split") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Loading clean data...")
clean_df = spark.read.parquet("../data/processed/clean_taxis.parquet")

# 2. TARGET PREPARATION (Filter Cash and create binary label)
# As seen in SQL, cash does not record tips. We keep only Credit Card (payment_type = 1)
modeling_df = clean_df.filter(col("payment_type") == 1)

# Create the target variable for Classification: 1 if tip exists, 0 otherwise
target_df = modeling_df.withColumn("has_tip", when(col("tip_amount") > 0, 1).otherwise(0))

print(f"Total useful trips for modeling (card only): {target_df.count()}")

# 3. OPTIMIZED STRATIFIED SPLIT (OOM Error-proof)
fixed_seed = 42
print("Executing optimized stratified split (70/15/15) with fixed seed...")

# Physically divide the majority and minority classes
pos_df = target_df.filter(col("has_tip") == 1)
neg_df = target_df.filter(col("has_tip") == 0)

# Apply randomSplit to each group separately
weights = [0.70, 0.15, 0.15]
train_pos, val_pos, test_pos = pos_df.randomSplit(weights, seed=fixed_seed)
train_neg, val_neg, test_neg = neg_df.randomSplit(weights, seed=fixed_seed)

# Union the equivalent pieces (Union operation is instantaneous in Spark)
train_df = train_pos.union(train_neg)
val_df = val_pos.union(val_neg)
test_df = test_pos.union(test_neg)

# 4. PROPORTION AND SIZE CHECK
print("\n=== SPLIT REPORT ===")
print(f"Train Size: {train_df.count()} records")
print(f"Validation Size: {val_df.count()} records")
print(f"Test Size: {test_df.count()} records")

# 5. SAVE TO PARQUET (Rubric: "Output: three datasets")
print("\nSaving partitions in Parquet format...")
train_df.write.mode("overwrite").parquet("../data/processed/train.parquet")
val_df.write.mode("overwrite").parquet("../data/processed/val.parquet")
test_df.write.mode("overwrite").parquet("../data/processed/test.parquet")

print("Datasets successfully saved and ready for Machine Learning!")

26/05/27 02:14:04 WARN Utils: Your hostname, LaptopPablo resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/27 02:14:04 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/27 02:14:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Loading clean data...


Total useful trips for modeling (card only): 7266200
Executing optimized stratified split (70/15/15) with fixed seed...

=== SPLIT REPORT ===


Train Size: 5087006 records


Validation Size: 1089424 records


Test Size: 1089770 records

Saving partitions in Parquet format...


Datasets successfully saved and ready for Machine Learning!
